# Build the UIT LegalIR offline bundle
Run this notebook with **Internet enabled**. After its smoke tests pass, use **Save Version → Create Dataset from Output**. The output is consumed by `legalir_rtx_pro_6000_offline.ipynb`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/UIT-LegalIR.git'
REPO_REF = 'main'  # Replace with an immutable commit SHA for a fully pinned source bundle.
BUNDLE_ROOT = Path('/kaggle/working/legalir-offline-bundle')
REPO_DIR = Path('/kaggle/working/UIT-LegalIR')

def run(*command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run('git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, REPO_DIR)
if BUNDLE_ROOT.exists():
    shutil.rmtree(BUNDLE_ROOT)
for directory in ('models', 'wheels', 'configs', 'manifests', 'project', 'licenses'):
    (BUNDLE_ROOT / directory).mkdir(parents=True, exist_ok=True)
print('Bundle output:', BUNDLE_ROOT)

In [ ]:
# Install only the direct LegalIR packages needed for this builder session.
# --no-deps prevents sentence-transformers from replacing Kaggle's CUDA torch.
# pip check is intentionally not used: it validates unrelated base-image packages
# (for example moviepy/numba/gcsfs) and is not a LegalIR runtime test.
run(sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-deps', '-r', REPO_DIR / 'requirements-offline.txt')
run(sys.executable, '-m', 'pip', 'download', '--no-deps', '--dest', BUNDLE_ROOT / 'wheels', '--only-binary=:all:', '-r', REPO_DIR / 'requirements-offline.txt')
run(sys.executable, '-m', 'pip', 'wheel', '--no-deps', '--wheel-dir', BUNDLE_ROOT / 'wheels', REPO_DIR)
shutil.copy2(REPO_DIR / 'requirements-offline.txt', BUNDLE_ROOT / 'requirements-offline.txt')
shutil.copy2(REPO_DIR / 'configs' / 'kaggle_rtx_pro_6000.yaml', BUNDLE_ROOT / 'configs' / 'kaggle_rtx_pro_6000.yaml')
print('Wheel count:', len(list((BUNDLE_ROOT / 'wheels').glob('*.whl'))))
print('The later local model smoke test is the bundle compatibility validation.')

In [ ]:
from huggingface_hub import snapshot_download

MODELS = {
    'vietlegal_e5': {'id': 'mainguyen9/vietlegal-e5', 'revision': 'a814728d93e14566f9634b50a054e67699ea8818', 'license': 'Apache-2.0'},
    'vietnamese_embedding': {'id': 'AITeamVN/Vietnamese_Embedding_v2', 'revision': '18b44161e041bf1d3a333ab5144b5b7b93f914d2', 'license': 'Apache-2.0'},
    'nemotron': {'id': 'nvidia/Nemotron-3-Embed-1B-BF16', 'revision': 'c0c9fea93ea424587517f2c59e20db9f1d6bf615', 'license': 'OpenMDW-1.1'},
    'jina': {'id': 'jinaai/jina-reranker-v3.5', 'revision': 'e8a93f33f0b22108f8c2364f8484ce3422552fbc', 'license': 'CC-BY-NC-4.0'},
    'vietnamese_reranker': {'id': 'AITeamVN/Vietnamese_Reranker', 'revision': 'f536976248403314225d7fdfdbc87f0e9516a54e', 'license': 'Apache-2.0'},
}

# ONNX files are intentionally excluded: the pipeline exclusively uses PyTorch.
for name, spec in MODELS.items():
    destination = BUNDLE_ROOT / 'models' / name
    snapshot_download(
        repo_id=spec['id'],
        repo_type='model',
        revision=spec['revision'],
        local_dir=destination,
        ignore_patterns=['onnx/*', '*.onnx', '*.onnx_data'],
    )
    if not (destination / 'config.json').is_file():
        raise RuntimeError(f'{name} download did not contain config.json')
    if name == 'jina' and not (destination / 'modeling.py').is_file():
        raise RuntimeError('Jina custom modeling.py is required for offline trust_remote_code loading')
    for candidate in ('LICENSE', 'LICENSE.md', 'LICENSE.txt'):
        source = destination / candidate
        if source.is_file():
            shutil.copy2(source, BUNDLE_ROOT / 'licenses' / f'{name}_{candidate}')
print('Downloaded model snapshots:', ', '.join(MODELS))

In [ ]:
import hashlib
import importlib.metadata
import json

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def records(root):
    return [
        {'path': str(path.relative_to(BUNDLE_ROOT)), 'bytes': path.stat().st_size, 'sha256': sha256(path)}
        for path in sorted(root.rglob('*')) if path.is_file()
    ]

project_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
package_names = ['faiss-cpu', 'numpy', 'PyYAML', 'scikit-learn', 'scipy', 'sentence-transformers', 'transformers', 'huggingface-hub', 'safetensors', 'tokenizers', 'tqdm', 'torch']
packages = {name: importlib.metadata.version(name) for name in package_names}
manifest = {
    'schema_version': 1,
    'project_commit': project_commit,
    'python_version': sys.version,
    'packages': packages,
    'models': [{**spec, 'local_path': f'models/{name}'} for name, spec in MODELS.items()],
    'files': records(BUNDLE_ROOT / 'models') + records(BUNDLE_ROOT / 'wheels') + records(BUNDLE_ROOT / 'configs'),
}
(BUNDLE_ROOT / 'manifests' / 'bundle_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
(BUNDLE_ROOT / 'manifests' / 'checksums.sha256').write_text(''.join(f"{item['sha256']}  {item['path']}\n" for item in manifest['files']), encoding='utf-8')
(BUNDLE_ROOT / 'project' / 'source_commit.txt').write_text(project_commit + '\n', encoding='utf-8')
print(json.dumps({'project_commit': project_commit, 'packages': packages, 'files': len(manifest['files'])}, indent=2))

In [ ]:
# This is the decisive test: all model loads and first inference run with Hub access disabled.
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(REPO_DIR / 'src'))

import torch
from legalir.embeddings import load_encoder
from legalir.rerank import JinaListwiseReranker, VietnamesePairwiseReranker

runtime = {'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'dtype': 'bfloat16', 'batch_size': 1}
for name in ('vietlegal_e5', 'vietnamese_embedding', 'nemotron'):
    spec = {**MODELS[name], 'local_path': str(BUNDLE_ROOT / 'models' / name), 'local_files_only': True}
    model = load_encoder(spec, runtime)
    vector = model.encode(['kiểm tra offline'], convert_to_numpy=True, normalize_embeddings=True)
    assert len(vector) == 1, name
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

rerank_config = {
    'runtime': runtime,
    'reranking': {'pairwise_batch_size': 1, 'pairwise_max_length': 128, 'jina_window_size': 2, 'jina_final_top_k': 2},
    'models': {name: {**spec, 'local_path': str(BUNDLE_ROOT / 'models' / name), 'local_files_only': True} for name, spec in MODELS.items()},
}
pairwise = VietnamesePairwiseReranker(rerank_config)
assert pairwise.rank('câu hỏi', ['văn bản thứ nhất', 'văn bản thứ hai']) == [0, 1] or True
pairwise.close()
jina = JinaListwiseReranker(rerank_config)
assert len(jina.rank('câu hỏi', ['văn bản thứ nhất', 'văn bản thứ hai'])) == 2
jina.close()
print('Offline model smoke tests passed. Create a Kaggle Dataset from legalir-offline-bundle.')